### 1) Install Required Packages

This cell installs the Python dependencies needed to run fine-tuned model benchmarking in this notebook environment.

In [ ]:
%pip install datasets transformers torch accelerate peft

### 2) Import Libraries

This cell imports dataset utilities, model/loading libraries, PEFT adapter support, and analysis/output dependencies.

In [ ]:
# Datasets
from datasets import load_dataset

# Output
import json
import pandas as pd

# Model
import torch
import re
import time
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

### 3) Prepare the Benchmark Dataset

This cell loads the GSM8K test split and applies the same filtering rule used in your benchmark notebooks for fair comparison.

In [ ]:
# Same benchmark subset setup as baseline notebook
dataset = load_dataset("gsm8k", "main", split="test")
dataset = dataset.filter(lambda x: len(x["answer"]) > 404)
print(dataset)

### 4) Load Base Model and LoRA Adapter

This cell loads the base Qwen model, attaches the selected fine-tuned LoRA adapter, and builds the generation pipeline.

In [ ]:
# Model and adapter setup
# Best available 1.5B LoRA run in this repo
BASE_MODEL = "Qwen/Qwen2-1.5B"
ADAPTER_PATH = "output/fine_tune/run_004_lr1e-4_r32_a64_ep3"

# Safety check: ensure adapter was trained from the same base model
with open(f"{ADAPTER_PATH}/adapter_config.json", "r", encoding="utf-8") as f:
    adapter_cfg = json.load(f)
expected_base = adapter_cfg.get("base_model_name_or_path")
if expected_base and expected_base != BASE_MODEL:
    raise ValueError(
        f"BASE_MODEL mismatch: adapter expects '{expected_base}', but got '{BASE_MODEL}'"
    )

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype="auto",
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

### 5) Define Prompt and Utility Functions

This cell defines helper functions for answer parsing, token counting, and prompt construction, aligned with `daniil_bm.ipynb` and `daniil_rl.ipynb`.

In [ ]:
# Helper function (aligned with daniil_bm / daniil_rl)
def extract_true_answer(answer):
    return answer.split("####")[-1].strip()

def extract_true_reasoning(answer):
    return answer.split("####")[0].strip()

def extract_predicted_answer(text):
    # take last number in output
    numbers = re.findall(r"-?\d+\.?\d*", text)
    return numbers[-1] if numbers else None

def count_tokens(input_text, output_text):
    input_tokens = len(tokenizer.encode(input_text))
    output_tokens = len(tokenizer.encode(output_text))
    return input_tokens, output_tokens

def build_prompt(question):
    return f"""You are a helpful math tutor.
Solve step by step and give the final answer.
Provide reasoning and intermediate steps and return the final numerical answer.
Output the final answer in the format on a new line: #### [final numeric answer]

Rules:
- Only provide reasoning and the final answer.
- Do not include any other text or explanations.
- Do not output code for reasoning, the reasoning must be a step by step process in natural language.
- Do not use , or . for formatting of large numbers (e.g. write 1000000 instead of 1,000,000).
- You must follow the exact format for the final answer.
- You must start directly with "Step 1:".
- Final line must be: [final numeric answer]

Response Format:
Step x: [reasoning step]
Step x + 1: [reasoning step]
...
Final Answer:[final numeric answer]

Question: {question}
Answer:"""

### 6) Run Benchmark Inference

This cell runs the fine-tuned model on the benchmark subset with two stochastic attempts per question, computes correctness/consistency, and records BM- and RL-compatible fields.

In [ ]:
# Benchmark run (aligned with daniil_bm; includes RL-compatible fields)
num_attempts = 2
results = []
consistent_count = 0

start_time = time.time()

for i, example in enumerate(dataset):
    question = example["question"]

    true_answer = extract_true_answer(example["answer"])
    true_reasoning = extract_true_reasoning(example["answer"])

    prompt = build_prompt(
        question=question,
    )

    model_responses = []
    model_answers = []
    model_correct = []
    model_input_tokens = []
    model_output_tokens = []
    model_reasonings = []

    # Run 2 attempts (temp=0.7)
    for attempt in range(num_attempts):
        output = pipe(
            prompt,
            max_new_tokens=320,
            return_full_text=False,
            do_sample=True,
            temperature=0.7,
            repetition_penalty=1.1,
        )[0]

        response = output["generated_text"]

        model_reasoning = output
        model_answer = extract_predicted_answer(response)

        try:
            is_correct = float(model_answer) == float(true_answer)
        except (TypeError, ValueError):
            is_correct = model_answer == true_answer

        input_tokens, output_tokens = count_tokens(prompt, response)

        # store per attempt
        model_answers.append(model_answer)
        model_correct.append(is_correct)
        model_input_tokens.append(input_tokens)
        model_output_tokens.append(output_tokens)
        model_reasonings.append(model_reasoning)
        model_responses.append(response)

    response_1 = model_responses[0]
    response_2 = model_responses[1]

    model_reasoning = model_reasonings[0]

    # Consistency (between the 2 attempts)
    cleaned_answers = [ans for ans in model_answers]

    is_consistent = (
        len(set(cleaned_answers)) == 1
        and all(ans not in ("", None) for ans in cleaned_answers)
    )

    if is_consistent:
        consistent_count += 1

    # Average metrics
    avg_accuracy = sum(model_correct) / num_attempts
    avg_input_tokens = sum(model_input_tokens) / num_attempts
    avg_output_tokens = sum(model_output_tokens) / num_attempts

    # RL-compatible single-attempt-style fields use attempt 1 values
    first_answer = model_answers[0]
    first_correct = model_correct[0]
    first_input_tokens = model_input_tokens[0]
    first_output_tokens = model_output_tokens[0]

    results.append({
        "question": question,
        "true_reasoning": true_reasoning,
        "true_answer": true_answer,
        "model_reasoning": model_reasoning,

        # per attempt (BM format)
        "model_answers": model_answers,
        "model_correct": model_correct,
        "model_input_tokens": model_input_tokens,
        "model_output_tokens": model_output_tokens,

        # aggregated (BM format)
        "avg_accuracy": avg_accuracy,
        "avg_input_tokens": avg_input_tokens,
        "avg_output_tokens": avg_output_tokens,
        "consistent": is_consistent,
        "response_1": response_1,
        "response_2": response_2,

        # RL-compatible fields
        "model_answer": first_answer,
        "correct": first_correct,
        "input_tokens": first_input_tokens,
        "output_tokens": first_output_tokens,

        "adapter_path": ADAPTER_PATH,
    })

end_time = time.time()

### 7) Save Fine-Tuned Results

This cell converts the collected outputs into a DataFrame, applies numeric casting for answers, and saves the benchmark results to `daniil_ft_results.json`.

In [ ]:
df = pd.DataFrame(results)
# Keep numeric conversion aligned with bm/rl notebooks
df["true_answer"] = pd.to_numeric(df["true_answer"], errors="coerce")
df["model_answer"] = pd.to_numeric(df["model_answer"], errors="coerce")

# Aligned naming for downstream comparison notebook
# (llm_as_a_judge.ipynb expects daniil_ft_results.json)
df.to_json("daniil_ft_results.json", orient="records", indent=4)
print("Saved: daniil_ft_results.json")

### 8) Report Final Metrics and Runtime

This cell prints overall benchmark statistics:
- total runtime and average runtime per example
- final accuracy
- average input/output tokens
- consistency score

In [ ]:
# Metrics summary
total_runtime = end_time - start_time
avg_runtime = total_runtime / len(results)
print(f"Total runtime for {len(results)} examples: {total_runtime:.3f} sec")
print(f"Average runtime per example: {avg_runtime:.3f} sec")

accuracy = sum(r["avg_accuracy"] for r in results) / len(results)
print("Final answer accuracy:", accuracy)

avg_input_tokens = sum(r["avg_input_tokens"] for r in results) / len(results)
avg_output_tokens = sum(r["avg_output_tokens"] for r in results) / len(results)
print("Avg input tokens:", avg_input_tokens)
print("Avg output tokens:", avg_output_tokens)

total_questions = len(results)
consistency_score = consistent_count / total_questions if total_questions > 0 else 0
print(f"Consistency score: {consistency_score:.4f}")